In [1]:
# ¿Cómo se expresan consultas sobre propinas como transformaciones MapReduce simples?
#
# archivo CSV ──► map / filter ──► reduce ──► resultados persistentes

import csv
from collections import defaultdict
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'submission').is_dir())
with (ROOT / 'data/tips.csv').open(encoding='utf-8', newline='') as file:
    rows = list(csv.DictReader(file))


In [2]:
# Map añade una métrica; filter conserva las cuentas que cumplen una condición de negocio.

rates = [{**row, 'tip_rate': round(float(row['tip']) / float(row['total_bill']), 6)} for row in rows]
dinner = [row for row in rows if row['time'] == 'Dinner']
dinner_large_tip = [row for row in dinner if float(row['tip']) > 5.0]
large_party = [row for row in rows if int(row['size']) >= 5 or float(row['total_bill']) > 45.0]


In [3]:
# Reduce agrupa las cuentas por sexo y acumula una unidad por observación.

counts = defaultdict(int)
for row in rows:
    counts[row['sex']] += 1
counts_by_sex = [{'sex': key, 'count': value} for key, value in sorted(counts.items())]


In [4]:
# Cada archivo persistente deja visible la respuesta de una consulta para el siguiente consumidor.

outputs = {'query_1_tip_rates.csv': (rates, list(rates[0])), 'query_2_dinner.csv': (dinner, list(rows[0])), 'query_3_dinner_large_tip.csv': (dinner_large_tip, list(rows[0])), 'query_4_large_party.csv': (large_party, list(rows[0])), 'query_5_count_by_sex.csv': (counts_by_sex, ['sex', 'count'])}
for name, (result, columns) in outputs.items():
    with (ROOT / 'submission' / name).open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=columns, lineterminator='\n')
        writer.writeheader()
        writer.writerows(result)
